In [1]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()


client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

In [4]:
def get_ai_answer(prompt):
    chat_completion = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=[
            {
                "role": "system",
                "content": prompt,
            }
        ],
    )
    response = chat_completion.choices[0].message.content
    return response

In [5]:
get_ai_answer("What is the weather in Bangalore?")

'I am sorry, I cannot provide up-to-date weather information as I am an AI language model and do not have access to real-time data. However, you can easily check the current weather in Bangalore by using a search engine, visiting a weather website, or checking a weather app on your smartphone.'

In [67]:
prompt = """
"Bob wants to know the current weather in his place at Bangalore.
He has the following 4 functions at hand:
1. def drinkWater(amount: int) -> None
2. def washClothes(temperature: int, detergent: str) -> None
3. def eatBurger(burgerType: str) -> None
4. def lookupWeather(location: str) -> None

Which of the above functions should he use and with what argument?

Output only a JSON response
eg:
{
  function_name : func_a
  function_arguments: [arg1, arg2]
}"
"""

In [64]:
response = get_ai_answer(prompt)
print(response)
print(type(response))

{
  "function_name" : "lookupWeather",
  "function_arguments": ["Bangalore"]
}
<class 'str'>


In [59]:
def get_current_weather(location, unit="fahrenheit"):
    print("inside the function")
    """Get the current weather in a given location"""
    if "tokyo" in location.lower():
        return json.dumps({"location": "Tokyo", "temperature": "10", "unit": unit})
    elif "san francisco" in location.lower():
        return json.dumps(
            {"location": "San Francisco", "temperature": "72", "unit": unit}
        )
    elif "paris" in location.lower():
        return json.dumps({"location": "Paris", "temperature": "22", "unit": unit})
    elif "bangalore" in location.lower():
        return json.dumps({"location": "Bangalore", "temperature": "67", "unit": unit})
    else:
        return json.dumps({"location": location, "temperature": "unknown"})


get_current_weather("tokyo")

inside the function


'{"location": "Tokyo", "temperature": "10", "unit": "fahrenheit"}'

In [41]:
import datetime


def get_current_time():
    current_time = datetime.datetime.now()
    return current_time


current_time = get_current_time()
print(current_time)

2024-02-08 00:51:14.878255


In [49]:
response_json = json.loads(response)
print(response_json)
function_to_be_called = response_json["function_name"]
function_args = response_json["function_arguments"]

print(function_to_be_called)
print(function_args)
print(type(function_to_be_called))
print(type(function_args))

{'function_name': 'lookupWeather', 'function_arguments': ['Bangalore']}
lookupWeather
['Bangalore']
<class 'str'>
<class 'list'>


In [50]:
function_map = {
    "lookupWeather": get_current_weather,
    "get_current_time": get_current_time,
}

In [51]:
function_to_be_called = function_map[function_to_be_called]
type(function_to_be_called)

function

In [60]:
function_to_be_called(*function_args)

'{"location": "Bangalore", "temperature": "unknown"}'

In [70]:
function_to_be_called("tokyo")

'{"location": "Tokyo", "temperature": "10", "unit": "fahrenheit"}'

In [68]:
new_prompt = f"""
User: "Bob wants to know the current weather in his place at Tokyo.
He has the following 4 functions at hand:
1. def drinkWater(amount: int) -> None
2. def washClothes(temperature: int, detergent: str) -> None
3. def eatBurger(burgerType: str) -> None
4. def lookupWeather(location: str) -> None

Which of the above functions should he use and with what argument?

Output only a JSON response
eg:
{{
  function_name : func_a
  function_arguments: [arg1, arg2]
}}"

---

AI: 
{{
  "function_name": "lookupWeather",
  "function_arguments": ["Bangalore"]
}}

---

User:
Function executed. 
got result: {function_to_be_called(*function_args)}

What is the answer to Bob's question?

---

AI:

"""

In [69]:
response = get_ai_answer(new_prompt)
print(response)

The current weather in Bob's place at Tokyo is unknown.
